In [ ]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Outline
0. Part 0 — Intro/Setup
1. Part 1 — OpenAI Python APIs
2. **Part 2 — OpenAI Agents SDK**
   * 02-00. Agents SDK Introduction: building blocks, ReAct pattern
   * 02-01. Single Agent System — Python Tutor: `Agent`, `Runner.run()`, `RunResult`
   * 02-02. **Conversation State in Agents: `previous_response_id`, `SQLiteSession`, `conversation_id`, `result.to_input_list()`**
   * 02-03. Streaming Text and Events: `Runner.run_streamed()`, `StreamEvent`, `run_item_stream_event`
   * 02-04. Python Tutor with a Model-Backed Input Guardrail: `@input_guardrail`, `GuardrailFunctionOutput`, `InputGuardrailTripwireTriggered`
   * 02-05. Tools
      * 02-05-00. Tools Overview: hosted, local, and custom tool taxonomy
      * 02-05-01. Financial Research Agent with a Custom Tool: `@function_tool`, hosted and custom tools
      * 02-05-02. Image Generation and Editing with `ImageGenerationTool`
      * 02-05-03. Multi-Agent Deitel Book Concierge: `FileSearchTool`, RAG, vector stores, handoffs
      * 02-05-04. Code Interpreter Tool: `CodeInterpreterTool`, hosted container
      * 02-05-05. Local MCP — SQLite Database: `MCPServerStdio`
      * 02-05-06. Hosted MCP — Weather and Geocoding: `MCPServerStreamableHttp`, MCP authentication
      * 02-05-07. AccuWeather Agent with `ComputerTool`: `AsyncComputer`, Playwright, `LocalBrowserComputer`
      * 02-05-08. `ShellTool` Folder Inspector: custom executor, human-in-the-loop approval
      * 02-05-09. Local LLM via LiteLLM and Ollama: `LitellmModel`
      * 02-05-10. Python Code Tutor with Dynamic Instructions: callable instructions, `RunContextWrapper`, typed context
3. Part 3 — Codex App
4. Part 4 — Wrap-Up and Additional References

---


# Creating Agents with the OpenAI Agents SDK
---

# Conversation State in Agents
* Persistence strategies for multi-turn agent conversations

---

## Four Options

| Option | Where state lives | What you pass on the next turn |
|---|---|---|
| `previous_response_id` | OpenAI Responses API | Last response ID + new user turn |
| `SQLiteSession` / `session` | SDK-managed local/custom store | Same session ID/store |
| `conversation_id` | OpenAI Conversations API | Same conversation ID + new user turn |
| `result.to_input_list()` | Your app memory | Prior input list + new user message |

* Pick one strategy per conversation
* `previous_response_id` for lightweight continuation
* `SQLiteSession` (or other similar techniques) when your app should own conversation history
* Apps incur **input token costs** for anything supplied with each request
---

## `previous_response_id`
* Lightweight server-managed response chaining
* Store the last model response ID and pass it back on the next turn
* With `auto_previous_response_id=True`, the SDK can automatically enable response chaining once a prior response exists
* Best for simple chat loops that pass only the new user input each turn
* Tradeoffs: **OpenAI-specific** and less explicit than a named Conversation resource
* Documentation:
    * [Agents SDK: server-managed conversations](https://openai.github.io/openai-agents-python/running_agents/#server-managed-conversations)
    * [Responses API: `previous_response_id`](https://platform.openai.com/docs/guides/conversation-state?api-mode=responses#passing-context-from-the-previous-response)
---

In [ ]:
"""A simple single-agent system that answers Python questions and
uses the OpenAI Agents SDK's built-in conversation management via
each response's previous_response_id."""
from agents import Agent, Runner, trace
from IPython.display import Markdown, display

# create an Agent
tutor = Agent(
    name="Python Tutor",
    model='gpt-5.5', # gpt-5.4-mini is the current default
    instructions="""You are a Python tutor for novice programmers.
        Provide concise, simple answers to Python questions."""
)

In [ ]:
print('I am a Python tutor. How can I help you?')
turn_count = 1 # turn counter for user-input prompts
previous_response_id = None # used to chain responses for context

# run until the user input is 'exit'
while ((prompt := input(f'Input [{turn_count}:] ')) != 'exit'):
    # launch agent asynchronously and await its response
    with trace(f'02-02-previous-response-id-[{turn_count}]'):
        result = await Runner.run(tutor, prompt,
            # SDK should chain responses for multi-turn conversation state
            previous_response_id=previous_response_id,
            auto_previous_response_id=True # support response chaining
        )

    display(Markdown(result.final_output)) # show answer from tutor agent
    turn_count += 1

    # save last response ID for context during the next turn
    previous_response_id = result.last_response_id

print('User terminated app.')

---
### Sample inputs:
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
* Are there any other ways?
---

## `SQLiteSession` / `session`
* Locally SDK-managed session memory
* Pass a session object into each run
* SDK loads prior conversation items before the run and stores new items after it
* Best for persistent chat state, resumable multi-turn apps, and app-owned memory
* **Tradeoff:** You operate the backing store and manage retention, compaction, privacy, and access controls
* **Works with non-OpenAI models too**
* Reference documentation:
    * [Sessions overview](https://openai.github.io/openai-agents-python/sessions/)
    * [SQLAlchemy sessions](https://openai.github.io/openai-agents-python/sessions/sqlalchemy_session/)
    * [Advanced SQLite sessions](https://openai.github.io/openai-agents-python/sessions/advanced_sqlite_session/)
    * [Encrypted sessions](https://openai.github.io/openai-agents-python/sessions/encrypted_session/)
---

In [ ]:
"""A simple single-agent system that answers Python questions and
stores conversation state using a local SQLiteSession."""
from agents import Agent, Runner, SQLiteSession, trace
from IPython.display import Markdown, display

tutor = Agent(
    name='Python Tutor',
    model='gpt-5.5',
    instructions="""You are a Python tutor for novice programmers.
        Provide concise, simple answers to Python questions."""
)

In [ ]:
# maintains an in-memory SQLite database for session state;
# add db_path parameter for databases that span executions
session = SQLiteSession('python_tutor_conversation')

print('I am a Python tutor. How can I help you?')
turn_count = 1

while (prompt := input(f'Input [{turn_count}]: ')) != 'exit':
    with trace('02-02-sqlite-session'):
        result = await Runner.run(tutor, prompt, session=session)

    display(Markdown(result.final_output))
    turn_count += 1

print('User terminated app.')

---
### Sample inputs:
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
* Are there any other ways?
---

## `conversation_id` 
* Long-lived, server-managed conversation resource
* Create an OpenAI Conversation once, then reuse its ID in agent runs
* Useful when several workers or services need the same durable server-side conversation
* Best for production systems using OpenAI models through the Responses API
* Context window limits apply per turn
    * For long-running conversations, compact, summarize, trim, or persist important state yourself
* Tradeoff: **OpenAI-managed and OpenAI-specific**
* Do not combine with SDK `session` in the same run
* Documentation:
    * [Conversation state guide](https://platform.openai.com/docs/guides/conversation-state?api-mode=responses)
    * [Agents SDK: state and conversation management](https://openai.github.io/openai-agents-python/running_agents/#state-and-conversation-management)

In [ ]:
"""A simple single-agent system that answers Python questions and
stores conversation state using the server-side Conversations API."""
from agents import Agent, Runner, trace
from openai import AsyncOpenAI
from IPython.display import Markdown, display

tutor = Agent(
    name='Python Tutor',
    model='gpt-5.5',
    instructions="""You are a Python tutor for novice programmers.
        Provide concise, simple answers to Python questions."""
)

client = AsyncOpenAI() # used to access Conversations API

# create the server-side conversation and get its ID
conversation = await client.conversations.create()
conversation_id = conversation.id

print('I am a Python tutor. How can I help you?')
turn_count = 1

while (prompt := input(f'Input [{turn_count}]: ')) != 'exit':
    with trace('02-02-conversation-id'):
        result = await Runner.run(
            tutor,
            prompt,
            conversation_id=conversation_id,
        )

    display(Markdown(result.final_output))
    turn_count += 1

print('User terminated app.')

---
### Sample inputs:
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
* Are there any other ways?
---

## `result.to_input_list()` 
* Manual, client-managed state
* Use the previous `RunResult` to create a replay-ready input list
* Append the next user message, then pass the full list into the next `Runner.run(...)`
* Best when your app needs to inspect, transform, trim, store, or audit exact conversation items
* Works with non-OpenAI models too
* Tradeoff: your app owns storage, trimming, persistence, privacy, token growth, and replay mechanics
* Documentation:
    * [Results: `to_input_list()`](https://openai.github.io/openai-agents-python/results/#input-next-turn-history-and-new-items)
    * [Agents SDK: state and conversation management](https://openai.github.io/openai-agents-python/running_agents/#state-and-conversation-management)

---

In [ ]:
"""A simple single-agent system that answers Python questions and
manually stores conversation state in a list."""
from agents import Agent, Runner, trace
from IPython.display import Markdown, display

tutor = Agent(
    name='Python Tutor',
    model='gpt-5.5',
    instructions="""You are a Python tutor for novice programmers.
        Provide concise, simple answers to Python questions."""
)

In [ ]:
print('I am a Python tutor. How can I help you?')
turn_count = 1
conversation = [] # we'll store conversation state here

while (prompt := input(f'Input [{turn_count}]: ')) != 'exit':
    conversation.append({'role': 'user', 'content': prompt})

    with trace('02-02-to-input-list'):
        result = await Runner.run(tutor, conversation)

    display(Markdown(result.final_output))
    conversation = result.to_input_list()
    turn_count += 1

print('User terminated app.')

---
### Sample inputs:
* Given a list containing 1-10, what's the easiest way to total them?
* Give me another way to do the same thing.
* Are there any other ways?
---

### Total Conversation from the Last Demo

In [ ]:
import json
print(json.dumps(conversation, indent=2))

## Important Constraints
* `SQLiteSession` / `session` and `result.to_input_list()` are **client-managed**
* `conversation_id` and `previous_response_id` are **OpenAI-managed** and apply when using OpenAI models through the Responses API
* Do **not** combine `session` with `conversation_id`, `previous_response_id`, or `auto_previous_response_id` in the same run
* `conversation_id` and `previous_response_id` are mutually exclusive
* Documentation:
    * [Running agents](https://openai.github.io/openai-agents-python/running_agents/)
    * [Results and state](https://openai.github.io/openai-agents-python/results/)
    * [Sessions](https://openai.github.io/openai-agents-python/sessions/)
---

## Comparison Table

| Option | Benefits | Disadvantages | Choose when... |
|---|---|---|---|
| `previous_response_id` | Lightest server-side continuation; pass only new user turn | Chain depends on last response ID; OpenAI-specific | You want simple Responses API continuation without creating conversations |
| `SQLiteSession` / `session` | Clean SDK API; automatic load/save; works with SQLite/custom stores | Client-managed persistence; store operations are your responsibility | You want SDK-managed local or app-owned memory |
| `conversation_id` | Durable server-side conversation; shareable across workers/services | OpenAI-specific; not combinable with `session`; less portable | You need a named conversation object for production OpenAI apps |
| `result.to_input_list()` | Explicit, provider-neutral, easy to inspect | You manually store, trim, replay, and secure history | You need full local control over history mechanics |


---
&copy; 1992-2026 by Deitel & Associates, Inc. and Pearson Education, Inc. All Rights Reserved.